# 02 — Prétraitement & Ingénierie des features

> **Objectif** : Transformer `data/raw/HelpDesk1.csv` en `data/processed/HelpDesk1.csv`
> prêt pour la modélisation ML et le moteur RAG.
>
> **Pipeline** :
> 1. Chargement des données brutes
> 2. Nettoyage textuel (`clean_text`)
> 3. Construction des features (`text_clean`, `text_rag`)
> 4. Analyse du prétraitement (avant/après, quasi-doublons)
> 5. Sauvegarde du fichier traité
> 6. Figures diagnostiques → `notebooks/figures/`

## 1. Imports & Configuration

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
print(f'Racine projet : {ROOT}')

import pandas as pd
import numpy as np
import yaml
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.rcParams['figure.dpi'] = 130
plt.style.use('seaborn-v0_8-whitegrid')

FIGURES_DIR = ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(ROOT / 'params.yaml', 'r') as f:
    params = yaml.safe_load(f)

print('params.yaml chargé')
print('Dossier figures :', FIGURES_DIR)

## 2. Chargement des données brutes

In [ ]:
raw_path = ROOT / params['data']['raw_path']
df = pd.read_csv(raw_path, encoding='utf-8-sig')

print(f'Dimensions         : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'Colonnes           : {list(df.columns)}')
print(f'Classes (Catégorie): {sorted(df["Catégorie"].unique())}')
print(f'\nDistribution des classes :')
print(df['Catégorie'].value_counts().to_string())
df.head(3)

In [ ]:
# Vérifications de qualité
print('=== Champs manquants (colonnes critiques) ===')
critical = ['Titre', 'Description', 'Catégorie', 'knowledge Base']
for col in critical:
    if col in df.columns:
        n = df[col].isnull().sum()
        print(f'  {col:<25}: {n} manquants ({n/len(df)*100:.1f}%)')

print('\n=== Cohérence des labels ===')
incoh = df.groupby('Titre')['Catégorie'].nunique()
print(f'  Titres avec plusieurs catégories : {(incoh > 1).sum()} (doit être 0)')

## 3. Nettoyage textuel — `clean_text`

La fonction `clean_text` (dans `src/data/preprocess.py`) :
- Normalise unicode (supprime accents)
- Supprime URLs, emails, IPs
- Supprime ponctuation
- Applique stop-words allégés (**termes IT discriminants conservés** : outlook, vpn, dns, sage…)
- Conserve tokens ≥ 2 caractères

La feature ML utilise **titre + description** (pas seulement la description) pour
capturer les termes discriminants souvent concentrés dans le titre.

In [ ]:
from data.preprocess import clean_text, build_text_for_model, build_text_for_rag

# Démonstration sur des cas représentatifs
exemples = [
    ('Problème de connexion VPN', 'Je n\'arrive plus à me connecter via VPN depuis ce matin.'),
    ('Outlook n\'envoie plus les mails', 'La messagerie Outlook affiche une erreur d\'authentification.'),
    ('Ransomware détecté sur le poste', 'Un ransomware a chiffré les fichiers du disque C:\\Users.'),
]

print(f'{"Titre":<35} {"→ text_clean"}')
print('-' * 80)
for titre, desc in exemples:
    cleaned = build_text_for_model(titre, desc)
    print(f'{titre:<35} → {cleaned}')

## 4. Construction des features ML et RAG

In [ ]:
# Feature ML : titre + description nettoyés
df['text_clean'] = df.apply(
    lambda r: build_text_for_model(str(r['Titre']), str(r['Description'])),
    axis=1
)

# Feature RAG : titre + description + résolution + knowledge base
df['text_rag'] = df.apply(
    lambda r: build_text_for_rag(
        description = str(r['Description']),
        resolution  = str(r.get('Résolution/Motif', '')),
        knowledge   = str(r.get('knowledge Base', '')),
        titre       = str(r['Titre']),
    ),
    axis=1
)

# Statistiques longueur
df['n_mots_brut']  = df['Description'].apply(lambda x: len(str(x).split()))
df['n_mots_clean'] = df['text_clean'].apply(lambda x: len(x.split()))

print('Features créées :')
print(f'  text_clean : {df["text_clean"].nunique()} uniques / {len(df)} tickets')
print(f'  text_rag   : {df["text_rag"].nunique()} uniques / {len(df)} tickets')
print()
print('Longueur médiane :')
print(f'  Brut (description) : {df["n_mots_brut"].median():.0f} mots')
print(f'  Nettoyé (titre+desc): {df["n_mots_clean"].median():.0f} mots')

## 4b. Simplification des colonnes catégorielles (features enrichies)

**Problème** : Priorité (4 valeurs) et Classification (5 valeurs) créent du bruit
pour le modèle — des distinctions trop fines qui ne corrèlent pas clairement avec
la catégorie IT à prédire.

**Solution** : regrouper en valeurs simplifiées et les ajouter comme **préfixe**
dans le texte d'entrée (`text_enriched`) → le modèle reçoit un signal structuré
supplémentaire sans changer la cible (`Catégorie`).

| Colonne | Avant (N valeurs) | Après (N valeurs) | Regroupement |
|---|---|---|---|
| **Priorité** | 4 | 3 | Majeur + Critique → **haute** |
| **Classification** | 5 | 3 | Incident + Alerte → **incident** ; Problème + Changement → **changement** |
| **Catégorie** | 8 | 8 | **inchangé** (cible métier) |

Format `text_enriched` : `"type <clf> priorite <prio> <titre> <description>"`

In [ ]:
from data.preprocess import simplify_priorite, simplify_classification, build_text_enriched

# ── Régroupements ────────────────────────────────────────────────────────────
df['priorite_simple']        = df['Priorité'].apply(simplify_priorite)
df['classification_simple']  = df['Classification'].apply(simplify_classification)

print('=== Priorité  4 -> 3 ===')
print(df.groupby(['Priorité','priorite_simple']).size().to_string())
print()
print('=== Classification  5 -> 3 ===')
print(df.groupby(['Classification','classification_simple']).size().to_string())

# ── Feature enrichie pour CamemBERT ─────────────────────────────────────────
df['text_enriched'] = df.apply(
    lambda r: build_text_enriched(
        titre          = str(r['Titre']),
        description    = str(r['Description']),
        classification = str(r['Classification']),
        priorite       = str(r['Priorité']),
    ),
    axis=1
)

print()
print(f'text_enriched : {df["text_enriched"].nunique()} uniques / {len(df)} tickets')
print()
print('Exemples text_enriched :')
for _, row in df.sample(3, random_state=42).iterrows():
    print(f'  [{row["Catégorie"]}] {row["text_enriched"][:90]}...')

## 5. Analyse de la redondance — structure du corpus

In [ ]:
n_total  = len(df)
n_unique = df['text_clean'].nunique()

print(f'Tickets totaux         : {n_total}')
print(f'Textes uniques         : {n_unique}')
print(f'Facteur de répétition  : {n_total/n_unique:.1f}× en moyenne')

print('\nTextes uniques par catégorie :')
stats_cat = df.groupby('Catégorie').agg(
    total   = ('text_clean', 'count'),
    uniques = ('text_clean', 'nunique'),
).assign(facteur=lambda x: (x['total'] / x['uniques']).round(1))
print(stats_cat.sort_values('total', ascending=False).to_string())

print('\n⚠️  Implication pour le split train/test :')
print('   → Utiliser grouper le decoupage par Titre, jamais ligne par ligne')
print('     pour éviter toute fuite de données.')

## 6. Sauvegarde du fichier prétraité

In [ ]:
processed_path = ROOT / params['data']['processed_path']
processed_path.parent.mkdir(parents=True, exist_ok=True)

# Supprimer les colonnes temporaires de calcul
cols_drop = ['n_mots_brut', 'n_mots_clean']
df_out = df.drop(columns=[c for c in cols_drop if c in df.columns])
df_out.to_csv(processed_path, index=False, encoding='utf-8-sig')

print(f'Fichier sauvegarde : {processed_path}')
print(f'   Lignes   : {len(df_out)}')
print(f'   Colonnes : {df_out.shape[1]}')
nouvelles = ['text_clean', 'text_rag', 'text_enriched', 'priorite_simple', 'classification_simple']
print(f'   Colonnes ajoutees : {[c for c in nouvelles if c in df_out.columns]}')

## 7. Figures diagnostiques

In [ ]:
# Figure 1 — Distribution des classes
fig, ax = plt.subplots(figsize=(10, 5))
counts = df['Catégorie'].value_counts()
colors = plt.cm.Set2.colors
bars = ax.barh(counts.index, counts.values,
               color=[colors[i % len(colors)] for i in range(len(counts))])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
ax.set_xlabel('Nombre de tickets', fontsize=11)
ax.set_title('Distribution des catégories — Dataset HelpDesk AI\n'
             f'(déséquilibre max/min : {counts.max()/counts.min():.1f}×)', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'prep_01_distribution_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ prep_01_distribution_classes.png')

In [ ]:
# Figure 2 — Longueur avant/après prétraitement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['n_mots_brut'], bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
med_brut = df['n_mots_brut'].median()
axes[0].axvline(med_brut, color='red', linestyle='--', label=f'Médiane = {med_brut:.0f}')
axes[0].set_title('Longueur brute (description)', fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].legend()

axes[1].hist(df['n_mots_clean'], bins=20, color='#55A868', edgecolor='white', alpha=0.85)
med_clean = df['n_mots_clean'].median()
axes[1].axvline(med_clean, color='red', linestyle='--', label=f'Médiane = {med_clean:.0f}')
axes[1].set_title('Longueur nettoyée (titre+desc)', fontweight='bold')
axes[1].set_xlabel('Nombre de mots')
axes[1].legend()

fig.suptitle('Impact du prétraitement sur la longueur des textes', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'prep_02_longueurs_avant_apres.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ prep_02_longueurs_avant_apres.png')

In [ ]:
# Figure 3 — Textes uniques vs total par catégorie
fig, ax = plt.subplots(figsize=(10, 5))
cats = stats_cat.sort_values('total', ascending=True)
y = range(len(cats))
ax.barh(y, cats['total'],   color='#4C72B0', alpha=0.6, label='Total tickets')
ax.barh(y, cats['uniques'], color='#C44E52', alpha=0.9, label='Textes uniques')
ax.set_yticks(list(y))
ax.set_yticklabels(cats.index)
ax.set_xlabel('Nombre de tickets')
ax.set_title('Diversité textuelle par catégorie\n'
             '(le chiffre à droite = facteur de répétition)', fontweight='bold')
ax.legend()
for i, (tot, uniq) in enumerate(zip(cats['total'], cats['uniques'])):
    ax.text(tot + 3, i, f'×{tot/uniq:.0f}', va='center', color='#4C72B0', fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'prep_03_redondance_par_categorie.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ prep_03_redondance_par_categorie.png')

In [ ]:
print('=' * 60)
print('RÉSUMÉ DU PRÉTRAITEMENT')
print('=' * 60)
print(f'  Tickets traités        : {len(df)}')
print(f'  text_clean             : titre + description nettoyés')
print(f'  text_rag               : titre + desc + résolution + KB')
print(f'  Textes uniques         : {n_unique}')
print(f'  Longueur médiane clean : {med_clean:.0f} mots')
print(f'  Fichier de sortie      : {processed_path.name}')
print(f'  Figures (3)            : notebooks/figures/prep_0*.png')
print()
print('  → Étape suivante : 03_modeling.ipynb')